# Notebook 05 — Population Clustering Preparation and Candidate Strategy Evaluation

Prepare the integrated structural population dataset for clustering and define the preprocessing strategies required for alternative clustering approaches.

This notebook focuses on transforming the structural population layer into analysis-ready representations suitable for different algorithm families, including:

- **K-Modes** for categorical-only clustering
- **K-Prototypes** for mixed categorical and numeric data
- **MCA + K-Means** for reduced-dimension clustering on indicator-encoded variables

## Objectives

- Load the integrated structural population dataset generated in Notebook 04
- Inspect feature types and prepare the clustering input space
- Define and document algorithm-specific preprocessing pipelines
- Produce comparable candidate representations for downstream clustering evaluation

This notebook prepares the candidate clustering design space.  
Model selection and final clustering decisions are handled in subsequent steps.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# Project configuration
# -------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
ACS_PATH = INTERIM_DIR / "acs_pums_5y"

FILE_NAME = "us_structural_population_v1.parquet"
DATA_PATH = ACS_PATH / FILE_NAME

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Integrated structural population dataset not found. "
        "Run notebook 04_household_features_and_population_merge first."
    )

print("=" * 80)
print("NOTEBOOK 05: POPULATION CLUSTERING PREPARATION")
print("=" * 80)
print("Input file:", DATA_PATH)

NOTEBOOK 05: POPULATION CLUSTERING PREPARATION
Input file: /Users/marcomagnolo/Projects/us-audience-segmentation/data/interim/acs_pums_5y/us_structural_population_v1.parquet


## Step 1: Load the Merged Structural Population Dataset

Load `mk_structural_population_layer_v1.parquet` with 19 columns representing the merged household (HUS) + person (PUS) structural features.

In [3]:
# Load the dataset

df = pd.read_parquet(DATA_PATH)

print("\n" + "=" * 80)
print("STEP 1: LOAD STRUCTURAL POPULATION DATASET")
print("=" * 80)

print(f"\n✓ Loaded: {DATA_PATH.name}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nColumns:")
print(list(df.columns))

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)


STEP 1: LOAD STRUCTURAL POPULATION DATASET

✓ Loaded: us_structural_population_v1.parquet
Dataset shape: 1,000,000 rows × 19 columns

Columns:
['serialno', 'sporder', 'pwgtp', 'actor_class', 'age_bin', 'sex_label', 'race_eth', 'edu_tier', 'emp_tier', 'income_tier_fixed', 'income_tier_pct', 'mar_tier', 'commute_tier', 'tenure', 'household_size', 'vehicle_count', 'puma', 'hhincome_tier', 'household_type']

First 5 rows:


,serialno,sporder,pwgtp,actor_class,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,income_tier_pct,mar_tier,commute_tier,tenure,household_size,vehicle_count,puma,hhincome_tier,household_type
0,2023HU1043211,2,58,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car,3,2,0,4316,0-19k,housing_unit
1,2019HU1076190,2,46,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P20-40,Never_Married,Car,3,4,2,5922,20-49k,housing_unit
2,2019GQ0046130,1,12,Adult,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,P0-20,Never_Married,NaN,group_quarters,1,0,11300,group_quarters,group_quarters
3,2019HU0403832,1,76,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Work_From_Home,1,5,2,2510,50-99k,housing_unit
4,2019HU0277198,1,64,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car,3,4,1,4607,20-49k,housing_unit



Data types:
serialno                  str
sporder                 int64
pwgtp                   int64
actor_class               str
age_bin                string
sex_label              string
race_eth               string
edu_tier               string
emp_tier                  str
income_tier_fixed      string
income_tier_pct           str
mar_tier               string
commute_tier         category
tenure                 string
household_size          int64
vehicle_count           Int64
puma                    int64
hhincome_tier          string
household_type            str
dtype: object


## Step 2: Inspect All 19 Columns

Examine summary statistics, missing values, and distributions for each column.

In [74]:

from pandas.api.types import (
    is_object_dtype,
    is_string_dtype,
    is_categorical_dtype,
    is_numeric_dtype
)

print("\n" + "=" * 80)
print("STEP 2: INSPECT ALL 19 COLUMNS")
print("=" * 80)

# Basic info
print("\nBasic Info:")
df.info()

# Missing values
print("\nMissing Values:")
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isna().sum().values,
    "Missing %": (df.isna().sum().values / len(df) * 100).round(2)
})
print(missing_summary.to_string(index=False))

# Unique values per column
print("\nUnique Values per Column:")
for col in df.columns:
    n_unique = df[col].nunique(dropna=True)
    dtype_str = str(df[col].dtype)
    print(f"  {col:25s} | Type: {dtype_str:12s} | Unique: {n_unique:6d}")

# Detailed examination of each column
print("\nDetailed Column Inspection:")
for col in df.columns:
    s = df[col]
    print(f"\n  {col}:")
    print(f"    Type: {s.dtype}")
    print(f"    Missing: {s.isna().sum()} ({s.isna().sum()/len(df)*100:.2f}%)")

    if is_object_dtype(s) or is_string_dtype(s) or is_categorical_dtype(s):
        vc = s.value_counts(dropna=False)
        print(f"    Unique values: {s.nunique(dropna=True)}")
        print(f"    Top 5: {vc.head().to_dict()}")
    elif is_numeric_dtype(s):
        print(f"    Min: {s.min()}, Max: {s.max()}, Mean: {s.mean():.2f}, Median: {s.median():.2f}")
    else:
        print("    Unhandled dtype - inspect manually")


STEP 2: INSPECT ALL 19 COLUMNS

Basic Info:
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   serialno           1000000 non-null  str     
 1   sporder            1000000 non-null  int64   
 2   pwgtp              1000000 non-null  int64   
 3   actor_class        1000000 non-null  str     
 4   age_bin            1000000 non-null  string  
 5   sex_label          1000000 non-null  string  
 6   race_eth           1000000 non-null  string  
 7   edu_tier           1000000 non-null  string  
 8   emp_tier           804465 non-null   str     
 9   income_tier_fixed  1000000 non-null  string  
 10  income_tier_pct    778466 non-null   str     
 11  mar_tier           1000000 non-null  string  
 12  commute_tier       473674 non-null   category
 13  tenure             1000000 non-null  string  
 14  household_size     1000000 non-nu

## Step 3: Define Clustering Feature Set

Select 14 features for clustering. Exclude identifiers (`serialno`, `sporder`) and weight column (`pwgtp`).

In [75]:
print("\n" + "=" * 80)
print("STEP 3: DEFINE CLUSTERING FEATURE SET")
print("=" * 80)

# Define features for clustering
clustering_features = [
    'age_bin', 'sex_label', 'race_eth', 'edu_tier', 'emp_tier',
    'income_tier_fixed', 'mar_tier', 'commute_tier', 'tenure',
    'household_size', 'vehicle_count', 'puma', 'hhincome_tier', 'household_type'
]

exclude_features = ['serialno', 'sporder', 'pwgtp']

print(f"\n✓ Clustering features ({len(clustering_features)}):")
for i, feat in enumerate(clustering_features, 1):
    if feat in df.columns:
        print(f"  {i:2d}. {feat:25s} ✓")
    else:
        print(f"  {i:2d}. {feat:25s} ✗ (NOT FOUND)")

print(f"\n✓ Excluded features ({len(exclude_features)}):")
for feat in exclude_features:
    if feat in df.columns:
        print(f"   - {feat:25s} ✓")
    else:
        print(f"   - {feat:25s} ✗ (NOT FOUND)")

# Verify all expected features exist
missing_from_data = [f for f in clustering_features if f not in df.columns]
if missing_from_data:
    print(f"\n⚠ WARNING: Missing features: {missing_from_data}")
else:
    print(f"\n✓ All {len(clustering_features)} clustering features present in dataset")

# Verify exclusions exist
missing_exclude = [f for f in exclude_features if f not in df.columns]
if missing_exclude:
    print(f"⚠ WARNING: Expected exclusions not found: {missing_exclude}")
else:
    print(f"✓ All {len(exclude_features)} exclusion features present")

# Get clustering data
df_cluster = df[clustering_features].copy()
print(f"\n✓ Clustering DataFrame shape: {df_cluster.shape}")
print(f"  {df_cluster.shape[0]:,} records × {df_cluster.shape[1]} features")


STEP 3: DEFINE CLUSTERING FEATURE SET

✓ Clustering features (14):
   1. age_bin                   ✓
   2. sex_label                 ✓
   3. race_eth                  ✓
   4. edu_tier                  ✓
   5. emp_tier                  ✓
   6. income_tier_fixed         ✓
   7. mar_tier                  ✓
   8. commute_tier              ✓
   9. tenure                    ✓
  10. household_size            ✓
  11. vehicle_count             ✓
  12. puma                      ✓
  13. hhincome_tier             ✓
  14. household_type            ✓

✓ Excluded features (3):
   - serialno                  ✓
   - sporder                   ✓
   - pwgtp                     ✓

✓ All 14 clustering features present in dataset
✓ All 3 exclusion features present

✓ Clustering DataFrame shape: (1000000, 14)
  1,000,000 records × 14 features


## Step 4: Handle Missing Values

Treat missing values as explicit "Missing" category for categorical features. Document the strategy for each feature.

In [76]:
print("\n" + "=" * 80)
print("STEP 4: HANDLE MISSING VALUES")
print("=" * 80)

# Define feature types EXPLICITLY based on clustering design
categorical_features = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "mar_tier",
    "commute_tier",
    "puma",
    "hhincome_tier",
    "household_type",
    "tenure"
]

numeric_features = [
    "household_size",
    "vehicle_count"
]

print("\nFeature Type Classification:")
print(f"  Categorical ({len(categorical_features)}): {categorical_features}")
print(f"  Numeric ({len(numeric_features)}): {numeric_features}")

# Missing value analysis
print("\nMissing Value Analysis by Feature:")
missing_strategy = {}

for feat in clustering_features:
    n_missing = df_cluster[feat].isna().sum()
    pct_missing = n_missing / len(df_cluster) * 100

    if n_missing == 0:
        strategy = "Complete - no action needed"
    elif feat in categorical_features:
        strategy = "Fill NaN with 'Missing' category where substantively appropriate"
    else:
        strategy = "Handle per algorithm (numeric imputation / exclusion / binning decision)"

    missing_strategy[feat] = {
        "missing_count": n_missing,
        "missing_pct": pct_missing,
        "strategy": strategy
    }

    print(f"  {feat:25s} | Missing: {n_missing:6d} ({pct_missing:6.2f}%) | {strategy}")

# Copy dataset
df_clean = df_cluster.copy()

# Fill categorical NaN with explicit Missing label
for feat in categorical_features:
    if df_clean[feat].isna().any():
        df_clean[feat] = df_clean[feat].astype("string").fillna("Missing")

filled_cats = [f for f in categorical_features if df_cluster[f].isna().any()]
print(f"\n✓ Categorical features filled: {filled_cats}")

# Numeric features: do not fill yet
numeric_with_na = [f for f in numeric_features if df_cluster[f].isna().any()]
print(f"✓ Numeric features with NA to handle per algorithm: {numeric_with_na}")

# Verification
print("\nVerification - remaining NaN count in df_clean:")
remaining_na = df_clean[clustering_features].isna().sum()
print(remaining_na[remaining_na > 0])


STEP 4: HANDLE MISSING VALUES

Feature Type Classification:
  Categorical (12): ['age_bin', 'sex_label', 'race_eth', 'edu_tier', 'emp_tier', 'income_tier_fixed', 'mar_tier', 'commute_tier', 'puma', 'hhincome_tier', 'household_type', 'tenure']
  Numeric (2): ['household_size', 'vehicle_count']

Missing Value Analysis by Feature:
  age_bin                   | Missing:      0 (  0.00%) | Complete - no action needed
  sex_label                 | Missing:      0 (  0.00%) | Complete - no action needed
  race_eth                  | Missing:      0 (  0.00%) | Complete - no action needed
  edu_tier                  | Missing:      0 (  0.00%) | Complete - no action needed
  emp_tier                  | Missing: 195535 ( 19.55%) | Fill NaN with 'Missing' category where substantively appropriate
  income_tier_fixed         | Missing:      0 (  0.00%) | Complete - no action needed
  mar_tier                  | Missing:      0 (  0.00%) | Complete - no action needed
  commute_tier              | 

## Step 5: Encoding Plan for K-Modes Clustering

**Algorithm**: K-Modes works with categorical data only. All features must be categorical (string/integer codes).

**Strategy**:
1. Keep all features categorical with explicit "Missing" category
2. Numeric features (tenure, household_size, vehicle_count) → Bin into categorical ranges
3. All features encoded as category labels (not binary indicators)

In [77]:
print("\n" + "=" * 80)
print("STEP 5: K-MODES ENCODING PLAN")
print("=" * 80)

# K-Modes plan
kmodes_plan = {}

# Categorical features kept as-is
categorical_kmodes = categorical_features.copy()
kmodes_plan["categorical_features"] = categorical_kmodes

print(f"\n✓ K-Modes: Categorical Features ({len(categorical_kmodes)}):")
for feat in categorical_kmodes:
    n_cats = df_clean[feat].nunique(dropna=True)
    sample_vals = df_clean[feat].dropna().astype(str).unique()[:3]
    print(f"  {feat:25s} | Categories: {n_cats:3d} | Values: {list(sample_vals)}...")

# Inspect tenure before deciding if it needs binning
print("\nTenure inspection:")
print(df_clean["tenure"].value_counts(dropna=False).head(20))

# Numeric features planned for binning in K-Modes
numeric_kmodes = {
    "household_size": "Bin into: Solo (1), Small (2-3), Medium (4-5), Large (6+)",
    "vehicle_count": "Bin into: None (0), One (1), Few (2-3), Many (4+)"
}

# Add tenure here only if confirmed numeric
kmodes_plan["numeric_binning"] = numeric_kmodes

print(f"\n✓ K-Modes: Numeric Features to Bin ({len(numeric_kmodes)}):")
for feat, strategy in numeric_kmodes.items():
    min_val = df_clean[feat].min()
    max_val = df_clean[feat].max()
    print(f"  {feat:25s} | Range: [{min_val}, {max_val}] | {strategy}")

print("\n✓ K-Modes Feature Set Summary:")
print(f"  Total features: {len(categorical_kmodes) + len(numeric_kmodes)}")
print("  Format: All categorical after binning")
print("  Dimensionality: One column per feature (no one-hot expansion)")
print("  Data matrix type: Pandas DataFrame of categorical/string features")

# ----------------------------------------------------
# Build K-Modes matrix
# ----------------------------------------------------

X_kmodes = df_clean[categorical_features + numeric_features].copy()

# Ensure categorical features are strings
for col in categorical_features:
    X_kmodes[col] = X_kmodes[col].astype(str)

# Binning functions
def bin_household_size(x):
    if pd.isna(x):
        return "Missing"
    elif x == 1:
        return "Solo"
    elif 2 <= x <= 3:
        return "Small"
    elif 4 <= x <= 5:
        return "Medium"
    else:
        return "Large"

def bin_vehicle_count(x):
    if pd.isna(x):
        return "Missing"
    elif x == 0:
        return "None"
    elif x == 1:
        return "One"
    elif 2 <= x <= 3:
        return "Few"
    else:
        return "Many"

# Apply binning
X_kmodes["household_size"] = X_kmodes["household_size"].apply(bin_household_size)
X_kmodes["vehicle_count"] = X_kmodes["vehicle_count"].apply(bin_vehicle_count)

# Ensure all columns are categorical/string
for col in X_kmodes.columns:
    X_kmodes[col] = X_kmodes[col].astype(str)

# Diagnostics
print("\nX_kmodes shape:")
print(X_kmodes.shape)

print("\nX_kmodes dtypes:")
print(X_kmodes.dtypes)

print("\nX_kmodes head:")
print(X_kmodes.head())


STEP 5: K-MODES ENCODING PLAN

✓ K-Modes: Categorical Features (12):
  age_bin                   | Categories:   9 | Values: ['35-44', '18-24', '65+']...
  sex_label                 | Categories:   2 | Values: ['Male', 'Female']...
  race_eth                  | Categories:   5 | Values: ['Black_NH', 'White_NH', 'Asian_NH']...
  edu_tier                  | Categories:   5 | Values: ['HS_or_less', 'Some_college', 'Bachelor']...
  emp_tier                  | Categories:   6 | Values: ['Employed', 'Other_Not_in_Labor_Force', 'Unemployed']...
  income_tier_fixed         | Categories:   7 | Values: ['0-19k', '20-49k', '<=0']...
  mar_tier                  | Categories:   3 | Values: ['Never_Married', 'Previously_Married', 'Married']...
  commute_tier              | Categories:   5 | Values: ['Car', 'Missing', 'Work_From_Home']...
  puma                      | Categories: 1150 | Values: ['4316', '5922', '11300']...
  hhincome_tier             | Categories:   6 | Values: ['0-19k', '20-49k', '

In [78]:
tenure_map = {
    "1": "Owner",
    "2": "Renter",
    "3": "No_Rent",
    "4": "Other",
    "group_quarters": "Group_Quarters"
}

df_clean["tenure"] = df_clean["tenure"].map(tenure_map)

In [79]:
df_clean.head()

,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,mar_tier,commute_tier,tenure,household_size,vehicle_count,puma,hhincome_tier,household_type
0,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,2,0,4316,0-19k,housing_unit
1,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,4,2,5922,20-49k,housing_unit
2,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,Never_Married,Missing,Group_Quarters,1,0,11300,group_quarters,group_quarters
3,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Work_From_Home,Owner,5,2,2510,50-99k,housing_unit
4,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,4,1,4607,20-49k,housing_unit


In [80]:
print(df_clean.isna().sum())

age_bin              0
sex_label            0
race_eth             0
edu_tier             0
emp_tier             0
income_tier_fixed    0
mar_tier             0
commute_tier         0
tenure               0
household_size       0
vehicle_count        0
puma                 0
hhincome_tier        0
household_type       0
dtype: int64


In [81]:
df_clean.nunique().sort_values(ascending=False)

puma                 1150
household_size         20
age_bin                 9
income_tier_fixed       7
vehicle_count           7
emp_tier                6
hhincome_tier           6
race_eth                5
edu_tier                5
commute_tier            5
tenure                  5
mar_tier                3
sex_label               2
household_type          2
dtype: int64

## Step 6: Encoding Plan for K-Prototypes Clustering

**Algorithm**: K-Prototypes handles both categorical and numeric features simultaneously.

**Strategy**:
1. Categorical features: Keep as category labels with "Missing" as explicit category
2. Numeric features: Keep as continuous values (handle NaN via imputation or median)
3. Mixed representation: Combine categorical + numeric in single feature matrix

In [82]:
print("\n" + "=" * 80)
print("STEP 6: K-PROTOTYPES ENCODING PLAN")
print("=" * 80)

# ----------------------------------------------------
# Explicit feature specification (no dependency on earlier variables)
# ----------------------------------------------------

categorical_features = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "mar_tier",
    "commute_tier",
    "tenure",
    "puma",
    "hhincome_tier",
    "household_type"
]

numeric_features = [
    "household_size",
    "vehicle_count"
]

# ----------------------------------------------------
# Feature split inspection
# ----------------------------------------------------

print("\n✓ K-Prototypes: Feature Split")

print(f"\n  Categorical ({len(categorical_features)}):")
for feat in categorical_features:
    n_cats = df_clean[feat].nunique(dropna=True)
    print(f"    - {feat:25s} | Categories: {n_cats}")

print(f"\n  Numeric ({len(numeric_features)}):")
for feat in numeric_features:
    min_val = df_clean[feat].min()
    max_val = df_clean[feat].max()
    mean_val = df_clean[feat].mean()
    n_missing = df_clean[feat].isna().sum()

    print(
        f"    - {feat:25s} | Range: [{min_val:.0f}, {max_val:.0f}] "
        f"| Mean: {mean_val:8.2f} | Missing: {n_missing}"
    )

# ----------------------------------------------------
# Numeric handling strategy
# ----------------------------------------------------

print("\n✓ K-Prototypes: Numeric Feature Handling")

for feat in numeric_features:
    median_val = df_clean[feat].median()
    print(f"    - {feat:25s} | Impute NaN with median ({median_val:.1f})")

# ----------------------------------------------------
# Build K-Prototypes matrix
# ----------------------------------------------------

X_kprototypes = df_clean[categorical_features + numeric_features].copy()

# enforce categorical types
for col in categorical_features:
    X_kprototypes[col] = X_kprototypes[col].astype(str)

# enforce numeric types
for col in numeric_features:
    X_kprototypes[col] = pd.to_numeric(X_kprototypes[col], errors="coerce")
    X_kprototypes[col] = X_kprototypes[col].fillna(X_kprototypes[col].median())

# ----------------------------------------------------
# Summary
# ----------------------------------------------------

print("\n✓ K-Prototypes Feature Set Summary:")
print(f"  Total features: {X_kprototypes.shape[1]}")
print(f"  Rows: {X_kprototypes.shape[0]}")
print(f"  Categorical columns: {len(categorical_features)}")
print(f"  Numeric columns: {len(numeric_features)}")
print("  Format: Mixed representation (categorical strings + numeric values)")
print("  Dimensionality: One column per feature (no expansion)")
print("  Data matrix type: Pandas DataFrame with mixed dtypes")

# ----------------------------------------------------
# Diagnostics
# ----------------------------------------------------

print("\nX_kprototypes shape:")
print(X_kprototypes.shape)

print("\nX_kprototypes dtypes:")
print(X_kprototypes.dtypes)

print("\nX_kprototypes head:")
print(X_kprototypes.head())


STEP 6: K-PROTOTYPES ENCODING PLAN

✓ K-Prototypes: Feature Split

  Categorical (12):
    - age_bin                   | Categories: 9
    - sex_label                 | Categories: 2
    - race_eth                  | Categories: 5
    - edu_tier                  | Categories: 5
    - emp_tier                  | Categories: 6
    - income_tier_fixed         | Categories: 7
    - mar_tier                  | Categories: 3
    - commute_tier              | Categories: 5
    - tenure                    | Categories: 5
    - puma                      | Categories: 1150
    - hhincome_tier             | Categories: 6
    - household_type            | Categories: 2

  Numeric (2):
    - household_size            | Range: [1, 20] | Mean:     3.30 | Missing: 0
    - vehicle_count             | Range: [0, 6] | Mean:     2.08 | Missing: 0

✓ K-Prototypes: Numeric Feature Handling
    - household_size            | Impute NaN with median (3.0)
    - vehicle_count             | Impute NaN with media

In [83]:
X_kprototypes["household_size"] = X_kprototypes["household_size"].astype("float64")
X_kprototypes["vehicle_count"] = X_kprototypes["vehicle_count"].astype("float64")

## Step 7: Encoding Plan for MCA + K-Means Clustering

**Algorithm**: Multiple Correspondence Analysis (MCA) converts categorical data to continuous dimensions, then K-Means clusters in the transformed space.

**Strategy**:
1. Convert all features to categorical with explicit "Missing" category
2. Bin numeric features into categorical ranges (same as K-Modes)
3. Apply MCA to generate continuous factorial axes (e.g., 10-20 dimensions)
4. Apply K-Means to the MCA coordinates

In [84]:
print("\n" + "=" * 80)
print("STEP 7: MCA + K-MEANS ENCODING PLAN")
print("=" * 80)

# ----------------------------------------------------
# Explicit feature specification
# ----------------------------------------------------

categorical_features = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "mar_tier",
    "commute_tier",
    "tenure",
    "puma",
    "hhincome_tier",
    "household_type"
]

numeric_features = [
    "household_size",
    "vehicle_count"
]

# ----------------------------------------------------
# Binning plans for numeric features
# ----------------------------------------------------

numeric_binning = {
    "household_size": {
        1: "Solo",
        (2, 3): "Small",
        (4, 5): "Medium",
        (6, float("inf")): "Large"
    },
    "vehicle_count": {
        0: "None",
        1: "One",
        (2, 3): "Few",
        (4, float("inf")): "Many"
    }
}

print("\n✓ MCA + K-Means: All Features as Categorical")

print(f"\n  Already categorical ({len(categorical_features)}):")
for feat in categorical_features:
    n_cats = df_clean[feat].nunique(dropna=True)
    print(f"    - {feat:25s} | Categories: {n_cats}")

print(f"\n  To bin (originally numeric, {len(numeric_features)}):")
for feat in numeric_features:
    min_val = df_clean[feat].min()
    max_val = df_clean[feat].max()
    print(f"    - {feat:25s} | Range: [{min_val:.0f}, {max_val:.0f}]")

# ----------------------------------------------------
# Build MCA input matrix
# ----------------------------------------------------

X_mca_input = df_clean[categorical_features + numeric_features].copy()

# Ensure categoricals are strings
for col in categorical_features:
    X_mca_input[col] = X_mca_input[col].astype(str)

# Bin numeric features into categories
def bin_household_size(x):
    if pd.isna(x):
        return "Missing"
    elif x == 1:
        return "Solo"
    elif 2 <= x <= 3:
        return "Small"
    elif 4 <= x <= 5:
        return "Medium"
    else:
        return "Large"

def bin_vehicle_count(x):
    if pd.isna(x):
        return "Missing"
    elif x == 0:
        return "None"
    elif x == 1:
        return "One"
    elif 2 <= x <= 3:
        return "Few"
    else:
        return "Many"

X_mca_input["household_size"] = X_mca_input["household_size"].apply(bin_household_size)
X_mca_input["vehicle_count"] = X_mca_input["vehicle_count"].apply(bin_vehicle_count)

# Convert all to string/category-like representation
for col in X_mca_input.columns:
    X_mca_input[col] = X_mca_input[col].astype(str)

# ----------------------------------------------------
# Indicator matrix estimation
# ----------------------------------------------------

total_categories = sum(X_mca_input[col].nunique(dropna=True) for col in X_mca_input.columns)
max_possible_axes = total_categories - X_mca_input.shape[1]
initial_mca_cap = min(max_possible_axes, 20)

print("\n✓ MCA: Indicator Matrix Estimation")
print(f"  Total categories across all features: ~{total_categories}")
print(f"  Maximum possible MCA axes: {max_possible_axes}")
print(f"  Initial MCA cap for testing: {initial_mca_cap}")
print("  Final retained dimensions will be chosen after fitting MCA based on explained inertia.")

print("\n✓ MCA + K-Means Feature Set Summary:")
print(f"  Total features: {X_mca_input.shape[1]}")
print(f"  Rows: {X_mca_input.shape[0]}")
print("  Input format: fully categorical (numeric features binned)")
print(f"  K-Means input later: MCA factorial coordinates")
print("  Advantage: dimensionality reduction + denoising of sparse indicator space")

print("\nX_mca_input shape:")
print(X_mca_input.shape)

print("\nX_mca_input dtypes:")
print(X_mca_input.dtypes)

print("\nX_mca_input head:")
print(X_mca_input.head())


STEP 7: MCA + K-MEANS ENCODING PLAN

✓ MCA + K-Means: All Features as Categorical

  Already categorical (12):
    - age_bin                   | Categories: 9
    - sex_label                 | Categories: 2
    - race_eth                  | Categories: 5
    - edu_tier                  | Categories: 5
    - emp_tier                  | Categories: 6
    - income_tier_fixed         | Categories: 7
    - mar_tier                  | Categories: 3
    - commute_tier              | Categories: 5
    - tenure                    | Categories: 5
    - puma                      | Categories: 1150
    - hhincome_tier             | Categories: 6
    - household_type            | Categories: 2

  To bin (originally numeric, 2):
    - household_size            | Range: [1, 20]
    - vehicle_count             | Range: [0, 6]

✓ MCA: Indicator Matrix Estimation
  Total categories across all features: ~1213
  Maximum possible MCA axes: 1199
  Initial MCA cap for testing: 20
  Final retained dimensions

In [85]:
# Short consolidated matrix for the 3 algos:
print("\n" + "=" * 80)
print("STEP 8: PREPARE FEATURE MATRICES")
print("=" * 80)

categorical_features = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "mar_tier",
    "commute_tier",
    "tenure",
    "puma",
    "hhincome_tier",
    "household_type"
]

numeric_features = [
    "household_size",
    "vehicle_count"
]

base_features = categorical_features + numeric_features

def bin_household_size(x):
    if pd.isna(x):
        return "Missing"
    elif x == 1:
        return "Solo"
    elif 2 <= x <= 3:
        return "Small"
    elif 4 <= x <= 5:
        return "Medium"
    else:
        return "Large"

def bin_vehicle_count(x):
    if pd.isna(x):
        return "Missing"
    elif x == 0:
        return "None"
    elif x == 1:
        return "One"
    elif 2 <= x <= 3:
        return "Few"
    else:
        return "Many"

# 8.1 K-Modes
X_kmodes = df_clean[base_features].copy()
for col in categorical_features:
    X_kmodes[col] = X_kmodes[col].astype(str)
X_kmodes["household_size"] = X_kmodes["household_size"].apply(bin_household_size).astype(str)
X_kmodes["vehicle_count"] = X_kmodes["vehicle_count"].apply(bin_vehicle_count).astype(str)

# 8.2 K-Prototypes
X_kprototypes = df_clean[base_features].copy()
for col in categorical_features:
    X_kprototypes[col] = X_kprototypes[col].astype(str)
for col in numeric_features:
    X_kprototypes[col] = pd.to_numeric(X_kprototypes[col], errors="coerce").astype("float64")

# 8.3 MCA input
X_mca_input = df_clean[base_features].copy()
for col in categorical_features:
    X_mca_input[col] = X_mca_input[col].astype(str)
X_mca_input["household_size"] = X_mca_input["household_size"].apply(bin_household_size).astype(str)
X_mca_input["vehicle_count"] = X_mca_input["vehicle_count"].apply(bin_vehicle_count).astype(str)

print("\nX_kmodes:", X_kmodes.shape)
print("X_kprototypes:", X_kprototypes.shape)
print("X_mca_input:", X_mca_input.shape)


STEP 8: PREPARE FEATURE MATRICES

X_kmodes: (1000000, 14)
X_kprototypes: (1000000, 14)
X_mca_input: (1000000, 14)


## Summary: Phase 1 Complete

### Dataset Loaded ✓
- **File**: mk_structural_population_layer_v1.parquet
- **Dimensions**: ~1M records × 19 columns
- **Clustering Features**: 14 selected (age_bin, sex_label, race_eth, edu_tier, emp_tier, income_tier_fixed, mar_tier, commute_tier, tenure, household_size, vehicle_count, puma, hhincome_tier, household_type)

### Preprocessing Strategy Defined ✓

**Three algorithm-specific feature matrices prepared:**

1. **X_kmodes**: All categorical, binned numerics → K-Modes clustering
2. **X_kprototypes**: Mixed categorical + numeric → K-Prototypes clustering  
3. **X_mca_input**: All categorical, binned numerics → MCA transformation → K-Means clustering

### Next Steps (Phase 2)
- Apply MCA to X_mca_input → generate continuous factorial axes
- Run K-Modes clustering on X_kmodes
- Run K-Prototypes clustering on X_kprototypes
- Run K-Means clustering on MCA coordinates
- Compare cluster quality across algorithms

# Phase 2 — Clusterability Diagnostics

Assess whether the structural population data exhibits clustering tendency using the Hopkins statistic on a representative sample.

In [86]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from scipy.spatial.distance import pdist, squareform

print("=" * 80)
print("PHASE 2: CLUSTERABILITY DIAGNOSTICS")
print("=" * 80)

PHASE 2: CLUSTERABILITY DIAGNOSTICS


## Step 1: Sample Data for Diagnostics

Draw a random sample of ~10,000 rows from X_mca_input for efficient computation.

In [87]:
# Sample ~10,000 rows for diagnostics
SAMPLE_SIZE = 10000
np.random.seed(42)  # For reproducibility

print("\n" + "=" * 80)
print("STEP 1: SAMPLE DATA FOR DIAGNOSTICS")
print("=" * 80)

# Draw random sample
sample_indices = np.random.choice(X_mca_input.shape[0], size=SAMPLE_SIZE, replace=False)
X_sample = X_mca_input.iloc[sample_indices].copy()

print(f"\n✓ Sampled {SAMPLE_SIZE:,} rows from X_mca_input")
print(f"  Original shape: {X_mca_input.shape}")
print(f"  Sample shape: {X_sample.shape}")
print(f"  Sample represents: {SAMPLE_SIZE/X_mca_input.shape[0]*100:.1f}% of full dataset")

# Verify sample is representative
print(f"\nSample composition check:")
for col in X_sample.columns[:5]:  # Check first 5 columns
    orig_dist = X_mca_input[col].value_counts(normalize=True).head(3)
    samp_dist = X_sample[col].value_counts(normalize=True).head(3)
    print(f"  {col}: Original top categories: {orig_dist.to_dict()}")
    print(f"        Sample top categories:   {samp_dist.to_dict()}")


STEP 1: SAMPLE DATA FOR DIAGNOSTICS

✓ Sampled 10,000 rows from X_mca_input
  Original shape: (1000000, 14)
  Sample shape: (10000, 14)
  Sample represents: 1.0% of full dataset

Sample composition check:
  age_bin: Original top categories: {'65+': 0.168362, '25-34': 0.136702, '35-44': 0.131023}
        Sample top categories:   {'65+': 0.1731, '25-34': 0.1395, '35-44': 0.1301}
  sex_label: Original top categories: {'Female': 0.50496, 'Male': 0.49504}
        Sample top categories:   {'Female': 0.5034, 'Male': 0.4966}
  race_eth: Original top categories: {'White_NH': 0.58175, 'Hispanic': 0.189931, 'Black_NH': 0.120245}
        Sample top categories:   {'White_NH': 0.5844, 'Hispanic': 0.1881, 'Black_NH': 0.1141}
  edu_tier: Original top categories: {'HS_or_less': 0.483884, 'Some_college': 0.231203, 'Bachelor': 0.156864}
        Sample top categories:   {'HS_or_less': 0.4849, 'Some_college': 0.2307, 'Bachelor': 0.16}
  emp_tier: Original top categories: {'Employed': 0.484026, 'Missing': 

## Step 2: One-Hot Encode Sampled Data

Convert categorical features to binary indicator matrix for distance computations.

In [88]:
print("\n" + "=" * 80)
print("STEP 2: ONE-HOT ENCODE SAMPLED DATA")
print("=" * 80)

# One-hot encode the sample
encoder = OneHotEncoder(sparse_output=False, drop=None)  # Keep all categories for diagnostics
X_encoded = encoder.fit_transform(X_sample)

print(f"\n✓ One-hot encoding complete")
print(f"  Input shape: {X_sample.shape}")
print(f"  Output shape: {X_encoded.shape}")
print(f"  Sparsity: {(X_encoded == 0).sum() / X_encoded.size * 100:.1f}% zeros")

# Get feature names
feature_names = encoder.get_feature_names_out(X_sample.columns)
print(f"  Total indicator features: {len(feature_names)}")

# Show some feature examples
print(f"\nExample encoded features:")
for i, name in enumerate(feature_names[:10]):
    print(f"  {i+1:2d}. {name}")
if len(feature_names) > 10:
    print(f"  ... and {len(feature_names) - 10} more")

# Memory check
encoded_size_mb = X_encoded.nbytes / (1024 * 1024)
print(f"\nMemory usage: {encoded_size_mb:.1f} MB")


STEP 2: ONE-HOT ENCODE SAMPLED DATA

✓ One-hot encoding complete
  Input shape: (10000, 14)
  Output shape: (10000, 1195)
  Sparsity: 98.8% zeros
  Total indicator features: 1195

Example encoded features:
   1. age_bin_0-5
   2. age_bin_13-17
   3. age_bin_18-24
   4. age_bin_25-34
   5. age_bin_35-44
   6. age_bin_45-54
   7. age_bin_55-64
   8. age_bin_6-12
   9. age_bin_65+
  10. sex_label_Female
  ... and 1185 more

Memory usage: 91.2 MB


## Step 3: Compute Hopkins Statistic

Calculate the Hopkins statistic to assess clustering tendency in the encoded data.

In [89]:
print("\n" + "=" * 80)
print("STEP 3: COMPUTE HOPKINS STATISTIC")
print("=" * 80)

from sklearn.neighbors import NearestNeighbors
import numpy as np

def hopkins_statistic(X, m=None):
    """
    Compute Hopkins statistic for clustering tendency.

    X : numeric matrix
    m : number of sample points (recommended ~5–10% of dataset)
    """

    X = np.array(X)
    n, d = X.shape

    if m is None:
        m = int(0.1 * n)

    np.random.seed(42)

    # Random sample of real points
    sample_indices = np.random.choice(n, m, replace=False)
    X_sample = X[sample_indices]

    # Fit nearest neighbor model
    nbrs = NearestNeighbors(n_neighbors=2).fit(X)

    # Distance from sampled real points to nearest neighbor
    distances_real, _ = nbrs.kneighbors(X_sample)
    u = distances_real[:, 1]   # skip distance to itself

    # Generate random points within data bounds
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)

    X_random = np.random.uniform(X_min, X_max, (m, d))

    distances_random, _ = nbrs.kneighbors(X_random)
    w = distances_random[:, 0]

    H = np.sum(w) / (np.sum(w) + np.sum(u))

    return H


# Compute Hopkins
hopkins_value = hopkins_statistic(X_encoded, m=1000)

print(f"\n✓ Hopkins statistic computed: {hopkins_value:.4f}")


STEP 3: COMPUTE HOPKINS STATISTIC

✓ Hopkins statistic computed: 0.9124


In [90]:
print("\n" + "=" * 80)
print("STEP 4: INTERPRETATION")
print("=" * 80)

if hopkins_value < 0.5:
    interpretation = "RANDOM / NOT CLUSTERABLE"
elif hopkins_value < 0.6:
    interpretation = "WEAK CLUSTERING"
elif hopkins_value < 0.7:
    interpretation = "MODERATE CLUSTERING"
elif hopkins_value < 0.8:
    interpretation = "GOOD CLUSTERING"
else:
    interpretation = "STRONG CLUSTERING"

print(f"\nHopkins Statistic: {hopkins_value:.4f}")
print(f"Interpretation: {interpretation}")

print("\nPHASE 2 COMPLETE")


STEP 4: INTERPRETATION

Hopkins Statistic: 0.9124
Interpretation: STRONG CLUSTERING

PHASE 2 COMPLETE


# Phase 3 — Clustering Experiments

Run K-Modes, K-Prototypes, and MCA+K-Means over a range of k values and collect costs and cluster sizes.

In [92]:
from kmodes.kmodes import KModes
from kmodes.kprototypes import KPrototypes
from sklearn.cluster import KMeans
import prince

print("\n" + "=" * 80)
print("PHASE 3: CLUSTERING EXPERIMENTS")
print("=" * 80)


PHASE 3: CLUSTERING EXPERIMENTS


## Step 1: K-Modes Clustering

Fit KModes on `X_kmodes` for k in 4..15, collect cost and cluster sizes.

In [93]:
kmodes_results = []
kmodes_models = {}

for k in range(4, 16):
    print(f"\nFitting K-Modes with k={k}...")
    model = KModes(n_clusters=k, init="Huang", n_init=5, verbose=0, random_state=42)
    clusters = model.fit_predict(X_kmodes)
    kmodes_models[k] = model
    # cost/inertia provided by model.cost_
    unique, counts = np.unique(clusters, return_counts=True)
    sizes = dict(zip(unique.tolist(), counts.tolist()))
    kmodes_results.append({
        "algorithm": "K-Modes",
        "k": k,
        "cost_or_inertia": model.cost_,
        "cluster_sizes": sizes
    })

print("\nFinished K-Modes runs.")


Fitting K-Modes with k=4...

Fitting K-Modes with k=5...

Fitting K-Modes with k=6...

Fitting K-Modes with k=7...

Fitting K-Modes with k=8...

Fitting K-Modes with k=9...

Fitting K-Modes with k=10...

Fitting K-Modes with k=11...

Fitting K-Modes with k=12...

Fitting K-Modes with k=13...

Fitting K-Modes with k=14...

Fitting K-Modes with k=15...

Finished K-Modes runs.


## Step 2: K-Prototypes Clustering

Prepare categorical column indices and run KPrototypes for each k.

In [94]:
kproto_results = []
kproto_models = {}

# categorical indices for X_kprototypes
cat_cols = [i for i,col in enumerate(X_kprototypes.columns) if col in categorical_features]

for k in range(4, 16):
    print(f"\nFitting K-Prototypes with k={k}...")
    model = KPrototypes(n_clusters=k, init="Huang", n_init=5, verbose=0, random_state=42)
    clusters = model.fit_predict(X_kprototypes, categorical=cat_cols)
    kproto_models[k] = model
    unique, counts = np.unique(clusters, return_counts=True)
    sizes = dict(zip(unique.tolist(), counts.tolist()))
    kproto_results.append({
        "algorithm": "K-Prototypes",
        "k": k,
        "cost_or_inertia": model.cost_
            if hasattr(model, 'cost_') else np.nan,
        "cluster_sizes": sizes
    })

print("\nFinished K-Prototypes runs.")


Fitting K-Prototypes with k=4...

Fitting K-Prototypes with k=5...

Fitting K-Prototypes with k=6...

Fitting K-Prototypes with k=7...

Fitting K-Prototypes with k=8...

Fitting K-Prototypes with k=9...

Fitting K-Prototypes with k=10...

Fitting K-Prototypes with k=11...

Fitting K-Prototypes with k=12...

Fitting K-Prototypes with k=13...

Fitting K-Prototypes with k=14...

Fitting K-Prototypes with k=15...

Finished K-Prototypes runs.


## Step 3: MCA + K-Means Clustering

One-hot encode `X_mca_input`, run MCA to 20 components, then K-Means for k=4..15.

In [95]:
kmeans_results = []
kmeans_models = {}

print("\nApplying MCA on categorical matrix...")

# Fit MCA directly on categorical dataframe
mca = prince.MCA(
    n_components=20,
    n_iter=3,
    random_state=42
)

X_mca_coords = mca.fit_transform(X_mca_input)

print(f"MCA coordinates shape: {X_mca_coords.shape}")

for k in range(4, 16):
    print(f"\nFitting K-Means on MCA coordinates with k={k}...")

    kmeans = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=42
    )

    clusters = kmeans.fit_predict(X_mca_coords)

    kmeans_models[k] = kmeans

    unique, counts = np.unique(clusters, return_counts=True)
    sizes = dict(zip(unique.tolist(), counts.tolist()))

    kmeans_results.append({
        "algorithm": "MCA+KMeans",
        "k": k,
        "cost_or_inertia": kmeans.inertia_,
        "cluster_sizes": sizes
    })

print("\nFinished MCA + KMeans runs.")


Applying MCA on categorical matrix...
MCA coordinates shape: (1000000, 20)

Fitting K-Means on MCA coordinates with k=4...

Fitting K-Means on MCA coordinates with k=5...

Fitting K-Means on MCA coordinates with k=6...

Fitting K-Means on MCA coordinates with k=7...

Fitting K-Means on MCA coordinates with k=8...

Fitting K-Means on MCA coordinates with k=9...

Fitting K-Means on MCA coordinates with k=10...

Fitting K-Means on MCA coordinates with k=11...

Fitting K-Means on MCA coordinates with k=12...

Fitting K-Means on MCA coordinates with k=13...

Fitting K-Means on MCA coordinates with k=14...

Fitting K-Means on MCA coordinates with k=15...

Finished MCA + KMeans runs.


## Step 4: Collect and Display Results

Combine results into one DataFrame and print summaries.

In [96]:
results = pd.DataFrame(kmodes_results + kproto_results + kmeans_results)

print("\nResults summary by algorithm and k:")
print(results[['algorithm','k','cost_or_inertia']].pivot(index='k', columns='algorithm', values='cost_or_inertia'))

print("\nCluster size distributions (sample):")
for alg in results['algorithm'].unique():
    print(f"\nAlgorithm: {alg}")
    subset = results[results['algorithm'] == alg]
    for _, row in subset.iterrows():
        print(f"  k={row['k']}: sizes={row['cluster_sizes']}")

print("\nPhase 3 complete - experiment results ready.")


Results summary by algorithm and k:
algorithm    K-Modes  K-Prototypes    MCA+KMeans
k                                               
4          5909214.0  6.131532e+06  1.551189e+06
5          5718394.0  5.770023e+06  1.452815e+06
6          5611932.0  5.568328e+06  1.383730e+06
7          5573280.0  5.389373e+06  1.343962e+06
8          5434535.0  5.213230e+06  1.310737e+06
9          5364130.0  5.106715e+06  1.281730e+06
10         5222617.0  4.978933e+06  1.259963e+06
11         5250443.0  4.922347e+06  1.240857e+06
12         5165516.0  4.809318e+06  1.225341e+06
13         5071515.0  4.695504e+06  1.211459e+06
14         5048845.0  4.656846e+06  1.198890e+06
15         4926499.0  4.606017e+06  1.188776e+06

Cluster size distributions (sample):

Algorithm: K-Modes
  k=4: sizes={0: 351403, 1: 232768, 2: 257670, 3: 158159}
  k=5: sizes={0: 242175, 1: 198359, 2: 194221, 3: 234152, 4: 131093}
  k=6: sizes={0: 232985, 1: 209129, 2: 151054, 3: 208339, 4: 124146, 5: 74347}
  k=7: sizes=

In [ ]:
# -------------------------------------------------------------------
# Save clustering models
# -------------------------------------------------------------------
import joblib

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models" / "clustering"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save K-Modes
for k, model in kmodes_models.items():
    joblib.dump(model, MODEL_DIR / f"kmodes_k{k}.joblib")

# Save K-Prototypes
for k, model in kproto_models.items():
    joblib.dump(model, MODEL_DIR / f"kprototypes_k{k}.joblib")

# Save MCA + KMeans
for k, model in kmeans_models.items():
    joblib.dump(model, MODEL_DIR / f"mca_kmeans_k{k}.joblib")

# Save MCA transformer
joblib.dump(mca, MODEL_DIR / "mca_transform.joblib")

print("Models saved to:", MODEL_DIR)


SAVE CLUSTERING MODELS
✓ Saved 12 K-Modes models
✓ Saved 12 K-Prototypes models
✓ Saved 12 MCA+KMeans models
✓ Saved MCA transformation

All clustering models saved to: clustering_models


In [ ]:
# -------------------------------------------------------------------
# Save clustering experiment results
# -------------------------------------------------------------------

RESULTS_PATH = PROJECT_ROOT / "outputs" / "tables" / "clustering_experiment_results.csv"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

results.to_csv(RESULTS_PATH, index=False)

print("Results saved to:", RESULTS_PATH)